# Genetic Mapping: From Poisson Distribution to Gene Ordering

**An Interactive Journey for Pattern Hunters**

This notebook demonstrates:
1. How Poisson distribution explains the 50% recombination limit
2. Two-point crosses and their limitations
3. Three-point crosses for distinguishing chromosome linkage
4. Determining gene order from recombination data

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
from scipy.special import factorial
import pandas as pd
from itertools import combinations

# Set style for better-looking plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## Part 1: The Poisson Distribution and Crossovers

### Understanding the "Shape of Uncertainty"

Crossovers during meiosis are **rare, random, independent events**. The number of crossovers in a chromosome region follows a **Poisson distribution**.

**Key Formula:**

$$P(k \text{ crossovers}) = \frac{e^{-\mu} \cdot \mu^k}{k!}$$

where μ (mu) = average number of crossovers = map distance in Morgans

In [ ]:
def plot_poisson_crossovers(mu_values):
    """
    Visualize Poisson distributions for different map distances
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, mu in enumerate(mu_values):
        ax = axes[idx]
        
        # Calculate probabilities for 0-15 crossovers
        k_values = np.arange(0, 16)
        probabilities = poisson.pmf(k_values, mu)
        
        # Separate odd and even
        odd_mask = k_values % 2 == 1
        even_mask = k_values % 2 == 0
        
        # Plot bars
        ax.bar(k_values[odd_mask], probabilities[odd_mask], 
               color='coral', alpha=0.7, label='Odd (Recombinant)', width=0.8)
        ax.bar(k_values[even_mask], probabilities[even_mask], 
               color='skyblue', alpha=0.7, label='Even (Parental)', width=0.8)
        
        # Calculate recombination frequency
        r_freq = np.sum(probabilities[odd_mask])
        
        ax.set_xlabel('Number of Crossovers (k)', fontsize=12)
        ax.set_ylabel('Probability', fontsize=12)
        ax.set_title(f'μ = {mu} Morgans ({mu*100:.0f} cM)\nRecombination Frequency = {r_freq*100:.2f}%', 
                    fontsize=13, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-0.5, 15.5)
    
    plt.tight_layout()
plt.savefig('poisson_crossovers.png', dpi=300, bbox_inches='tight')
plt.show()

# Demonstrate with different map distances
plot_poisson_crossovers([0.2, 0.5, 1.0, 2.0])

### Key Observation:

Notice how as μ increases (genes farther apart):
- The distribution spreads out
- Odd and even crossovers become more balanced
- **Recombination frequency approaches 50% but NEVER exceeds it!**

In [ ]:
def calculate_recombination_frequency(mu):
    """
    Calculate recombination frequency from map distance using Poisson
    r = (1 - e^(-2μ)) / 2
    """
    return (1 - np.exp(-2 * mu)) / 2

def haldane_mapping_function(r):
    """
    Convert recombination frequency back to map distance (Haldane's function)
    μ = -0.5 * ln(1 - 2r)
    """
    return -0.5 * np.log(1 - 2 * r)

# Create the asymptotic approach to 50%
mu_range = np.linspace(0, 3, 300)
r_values = calculate_recombination_frequency(mu_range)

plt.figure(figsize=(12, 7))
plt.plot(mu_range * 100, r_values * 100, linewidth=3, color='darkblue', label='Recombination Frequency')
plt.axhline(y=50, color='red', linestyle='--', linewidth=2, label='50% Limit', alpha=0.7)
plt.axhline(y=49, color='orange', linestyle=':', linewidth=1.5, alpha=0.5)

# Add annotations
plt.annotate('Approaches 50%\nbut never exceeds!', 
            xy=(200, 49), xytext=(150, 42),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=12, fontweight='bold', color='red')

plt.xlabel('Map Distance (centiMorgans)', fontsize=13)
plt.ylabel('Recombination Frequency (%)', fontsize=13)
plt.title('The 50% Recombination Limit: Poisson Distribution Guarantee', fontsize=15, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(0, 300)
plt.ylim(0, 52)
plt.tight_layout()
plt.savefig('50percent_limit.png', dpi=300, bbox_inches='tight')
plt.show()

# Show some specific values
print("\n" + "="*60)
print("Map Distance → Recombination Frequency Conversion")
print("="*60)
test_distances = [5, 10, 20, 50, 100, 150, 200]
for dist_cM in test_distances:
    mu = dist_cM / 100  # Convert cM to Morgans
    r = calculate_recombination_frequency(mu)
    print(f"{dist_cM:3d} cM → {r*100:6.2f}% recombination")
print("="*60)

## Part 2: Two Causes of 50% Recombination

### Cause 1: Genes Far Apart on SAME Chromosome
- Multiple crossovers (Poisson distribution)
- **Approaches** 50% asymptotically

### Cause 2: Genes on DIFFERENT Chromosomes
- Independent assortment (Mendel's 2nd Law)
- **Exactly** 50% always

**The Problem**: You can't distinguish these just from recombination frequency!

In [ ]:
def simulate_two_point_cross(scenario, n_offspring=1000):
    """
    Simulate a two-point cross
    scenario: 'linked_close', 'linked_far', or 'unlinked'
    """
    np.random.seed(42)
    
    if scenario == 'linked_close':
        # 10 cM apart - true linkage
        r = 0.10
        title = "Linked (10 cM apart on same chromosome)"
    elif scenario == 'linked_far':
        # 100 cM apart - far on same chromosome
        mu = 1.0
        r = calculate_recombination_frequency(mu)
        title = f"Linked but distant (100 cM, ~{r*100:.1f}% recombination)"
    else:  # unlinked
        r = 0.50
        title = "Unlinked (different chromosomes)"
    
    # Generate offspring
    recombinants = np.random.random(n_offspring) < r
    
    # Count gamete types (assuming AaBb x aabb testcross)
    parental = np.sum(~recombinants)
    recombinant = np.sum(recombinants)
    
    return {
        'title': title,
        'parental': parental,
        'recombinant': recombinant,
        'r_observed': recombinant / n_offspring,
        'r_expected': r
    }

# Simulate all three scenarios
scenarios = ['linked_close', 'linked_far', 'unlinked']
results = [simulate_two_point_cross(s) for s in scenarios]

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, result in enumerate(results):
    ax = axes[idx]
    
    categories = ['Parental\nType', 'Recombinant\nType']
    counts = [result['parental'], result['recombinant']]
    colors = ['skyblue', 'coral']
    
    bars = ax.bar(categories, counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    
    # Add count labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\n({count/10:.1f}%)',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_ylabel('Number of Offspring', fontsize=12)
    ax.set_title(f"{result['title']}\nObserved RF = {result['r_observed']*100:.1f}%", 
                fontsize=11, fontweight='bold')
    ax.set_ylim(0, 600)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('two_point_cross_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("TWO-POINT CROSS PROBLEM: Can't distinguish linkage from independence!")
print("="*70)
for result in results:
    print(f"\n{result['title']}")
    print(f"  Observed RF: {result['r_observed']*100:.1f}%")
    if result['r_observed'] > 0.45:
        print("  ⚠️  Could be EITHER far apart on same chromosome OR different chromosomes!")
print("\n" + "="*70)

## Part 3: Three-Point Cross - The Solution!

### Why Three Points?

By testing THREE genes simultaneously, we can:
1. Determine if all three are on the same chromosome
2. Figure out their **linear order**
3. Calculate accurate map distances

### The Logic:

If genes A, B, and C are on the same chromosome in that order:
```
A -------- B -------- C
   d(AB)       d(BC)
```

Then: **d(AC) ≈ d(AB) + d(BC)**  (with slight correction for double crossovers)

If they're on different chromosomes, this relationship breaks down!

In [ ]:
def simulate_three_point_cross(gene_positions, n_offspring=1000):
    """
    Simulate a three-point testcross
    gene_positions: dict with 'A', 'B', 'C' positions in cM
    Example: {'A': 0, 'B': 15, 'C': 35} means A-B = 15 cM, B-C = 20 cM
    """
    np.random.seed(42)
    
    # Calculate map distances
    positions = sorted(gene_positions.items(), key=lambda x: x[1])
    gene_order = [g[0] for g in positions]
    
    d_AB = abs(gene_positions['A'] - gene_positions['B'])
    d_BC = abs(gene_positions['B'] - gene_positions['C'])
    d_AC = abs(gene_positions['A'] - gene_positions['C'])
    
    # Convert to recombination frequencies (accounting for multiple crossovers)
    r_AB = calculate_recombination_frequency(d_AB / 100)
    r_BC = calculate_recombination_frequency(d_BC / 100)
    r_AC = calculate_recombination_frequency(d_AC / 100)
    
    # Simulate offspring
    offspring = []
    
    for _ in range(n_offspring):
        # Determine recombination for each interval
        recomb_AB = np.random.random() < r_AB
        recomb_BC = np.random.random() < r_BC
        recomb_AC = np.random.random() < r_AC
        
        offspring.append({
            'AB': recomb_AB,
            'BC': recomb_BC,
            'AC': recomb_AC
        })
    
    # Calculate observed recombination frequencies
    obs_r_AB = sum(o['AB'] for o in offspring) / n_offspring
    obs_r_BC = sum(o['BC'] for o in offspring) / n_offspring
    obs_r_AC = sum(o['AC'] for o in offspring) / n_offspring
    
    return {
        'gene_order': gene_order,
        'true_distances': {'AB': d_AB, 'BC': d_BC, 'AC': d_AC},
        'expected_rf': {'AB': r_AB, 'BC': r_BC, 'AC': r_AC},
        'observed_rf': {'AB': obs_r_AB, 'BC': obs_r_BC, 'AC': obs_r_AC},
        'offspring': offspring
    }

# Example 1: Three genes on same chromosome
print("\n" + "="*70)
print("SCENARIO 1: Three genes on the SAME chromosome")
print("="*70)

linked_genes = {'A': 0, 'B': 15, 'C': 35}  # A----15cM----B----20cM----C
result_linked = simulate_three_point_cross(linked_genes)

print(f"\nTrue gene order: {' - '.join(result_linked['gene_order'])}")
print(f"True map distances: A-B = {result_linked['true_distances']['AB']:.0f} cM, "
      f"B-C = {result_linked['true_distances']['BC']:.0f} cM, "
      f"A-C = {result_linked['true_distances']['AC']:.0f} cM")

print("\nObserved Recombination Frequencies:")
print(f"  A-B: {result_linked['observed_rf']['AB']*100:.1f}%")
print(f"  B-C: {result_linked['observed_rf']['BC']*100:.1f}%")
print(f"  A-C: {result_linked['observed_rf']['AC']*100:.1f}%")

print("\n✓ Notice: d(AC) ≈ d(AB) + d(BC) — consistent with linear arrangement!")

# Example 2: Genes on different chromosomes
print("\n" + "="*70)
print("SCENARIO 2: Genes on DIFFERENT chromosomes")
print("="*70)

# Simulate by making all distances very large
unlinked_genes = {'A': 0, 'B': 200, 'C': 400}  # All effectively unlinked
result_unlinked = simulate_three_point_cross(unlinked_genes)

print("\nObserved Recombination Frequencies:")
print(f"  A-B: {result_unlinked['observed_rf']['AB']*100:.1f}%")
print(f"  B-C: {result_unlinked['observed_rf']['BC']*100:.1f}%")
print(f"  A-C: {result_unlinked['observed_rf']['AC']*100:.1f}%")

print("\n✓ Notice: All ~50% — indicates independent assortment!")
print("✓ The additivity rule BREAKS DOWN — not on same chromosome!")
print("="*70)

## Part 4: Determining Gene Order from Real Data

### The Algorithm:

Given recombination frequencies between three genes:
1. Calculate RF for all three pairs: AB, AC, BC
2. The **largest RF** indicates the two genes on the **ends**
3. The **middle gene** is the one that shows intermediate RFs with both end genes
4. Verify: RF(ends) ≈ RF(one end to middle) + RF(middle to other end)

In [ ]:
def determine_gene_order_from_rf(rf_data):
    """
    Determine gene order from recombination frequency data
    rf_data: dict like {'AB': 0.15, 'AC': 0.35, 'BC': 0.20}
    Returns: gene order and map
    """
    # Find the pair with maximum RF (these are the outer genes)
    max_pair = max(rf_data.items(), key=lambda x: x[1])
    outer_genes = list(max_pair[0])
    max_rf = max_pair[1]
    
    # Identify the middle gene
    all_genes = set('ABC')
    middle_gene = list(all_genes - set(outer_genes))[0]
    
    # Get RFs to middle gene
    pair1 = ''.join(sorted([outer_genes[0], middle_gene]))
    pair2 = ''.join(sorted([outer_genes[1], middle_gene]))
    
    rf1 = rf_data[pair1]
    rf2 = rf_data[pair2]
    
    # Determine which outer gene is on which side
    if rf1 < rf2:
        gene_order = [outer_genes[0], middle_gene, outer_genes[1]]
        distances = [rf1 * 100, rf2 * 100]
    else:
        gene_order = [outer_genes[1], middle_gene, outer_genes[0]]
        distances = [rf2 * 100, rf1 * 100]
    
    # Verify additivity
    expected_outer_rf = (distances[0] + distances[1]) / 100
    # Correct for multiple crossovers
    expected_outer_rf = calculate_recombination_frequency((distances[0] + distances[1]) / 100)
    
    return {
        'order': gene_order,
        'distances': distances,
        'total_distance': sum(distances),
        'observed_outer_rf': max_rf,
        'expected_outer_rf': expected_outer_rf,
        'additivity_check': abs(max_rf - expected_outer_rf) < 0.05
    }

# Example problem sets
print("\n" + "="*70)
print("GENE ORDERING PRACTICE PROBLEMS")
print("="*70)

# Problem 1
print("\nProblem 1: Labeo rohita (Indian major carp) microsatellite markers")
print("-" * 70)
problem1 = {
    'AB': 0.12,  # 12% recombination
    'AC': 0.28,  # 28% recombination
    'BC': 0.16   # 16% recombination
}

print("Given recombination frequencies:")
for pair, rf in problem1.items():
    print(f"  {pair}: {rf*100:.0f}%")

solution1 = determine_gene_order_from_rf(problem1)
print(f"\n✓ Gene order: {' - '.join(solution1['order'])}")
print(f"✓ Map distances: {solution1['distances'][0]:.1f} cM - {solution1['distances'][1]:.1f} cM")
print(f"✓ Total map length: {solution1['total_distance']:.1f} cM")
print(f"✓ Additivity check: {'PASS' if solution1['additivity_check'] else 'FAIL'}")

# Problem 2
print("\n" + "-" * 70)
print("Problem 2: Earthworm (Metaphire) molecular markers from mining regions")
print("-" * 70)
problem2 = {
    'AB': 0.22,
    'AC': 0.38,
    'BC': 0.16
}

print("Given recombination frequencies:")
for pair, rf in problem2.items():
    print(f"  {pair}: {rf*100:.0f}%")

solution2 = determine_gene_order_from_rf(problem2)
print(f"\n✓ Gene order: {' - '.join(solution2['order'])}")
print(f"✓ Map distances: {solution2['distances'][0]:.1f} cM - {solution2['distances'][1]:.1f} cM")
print(f"✓ Total map length: {solution2['total_distance']:.1f} cM")
print(f"✓ Additivity check: {'PASS' if solution2['additivity_check'] else 'FAIL'}")

print("\n" + "="*70)

## Part 5: Visualizing Gene Maps

In [ ]:
def draw_gene_map(gene_order, distances, title):
    """
    Draw a linear genetic map
    """
    fig, ax = plt.subplots(figsize=(14, 3))
    
    # Calculate cumulative positions
    positions = [0]
    for d in distances:
        positions.append(positions[-1] + d)
    
    # Draw chromosome line
    total_length = sum(distances)
    ax.plot([0, total_length], [0.5, 0.5], 'k-', linewidth=4, solid_capstyle='round')
    
    # Draw genes
    colors = ['red', 'blue', 'green']
    for i, (gene, pos) in enumerate(zip(gene_order, positions)):
        # Gene marker
        ax.plot([pos, pos], [0.3, 0.7], linewidth=3, color=colors[i])
        # Gene label
        ax.text(pos, 0.85, f"Gene {gene}", ha='center', fontsize=13, 
               fontweight='bold', color=colors[i])
        # Position label
        ax.text(pos, 0.15, f"{pos:.1f} cM", ha='center', fontsize=11)
    
    # Draw distance annotations
    for i in range(len(distances)):
        mid_pos = (positions[i] + positions[i+1]) / 2
        ax.annotate('', xy=(positions[i+1], 0.5), xytext=(positions[i], 0.5),
                   arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=2))
        ax.text(mid_pos, 0.55, f"{distances[i]:.1f} cM", ha='center', 
               fontsize=11, fontweight='bold', color='darkgreen',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax.set_xlim(-5, total_length + 5)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(title, fontsize=15, fontweight='bold', pad=20)
    
    return fig

# Draw maps for our examples
fig1 = draw_gene_map(solution1['order'], solution1['distances'], 
                     'Genetic Map: Labeo rohita Microsatellite Markers')
plt.savefig('gene_map_labeo.png', dpi=300, bbox_inches='tight')
plt.show()

fig2 = draw_gene_map(solution2['order'], solution2['distances'],
                     'Genetic Map: Earthworm Markers (Mining Region)')
plt.savefig('gene_map_earthworm.png', dpi=300, bbox_inches='tight')
plt.show()

## Part 6: Interactive Exercise - Order These Genes!

Try these practice problems:

In [ ]:
def create_practice_problem(name, description):
    """
    Generate a random three-point cross problem
    """
    # Generate random gene positions
    pos_A = 0
    pos_B = np.random.randint(10, 30)
    pos_C = np.random.randint(pos_B + 10, pos_B + 40)
    
    # Shuffle to randomize which is which
    positions = {'A': pos_A, 'B': pos_B, 'C': pos_C}
    
    # Calculate RFs
    d_AB = abs(positions['A'] - positions['B'])
    d_BC = abs(positions['B'] - positions['C'])
    d_AC = abs(positions['A'] - positions['C'])
    
    rf_AB = calculate_recombination_frequency(d_AB / 100)
    rf_BC = calculate_recombination_frequency(d_BC / 100)
    rf_AC = calculate_recombination_frequency(d_AC / 100)
    
    print(f"\n{'='*70}")
    print(f"Practice Problem: {name}")
    print(f"{description}")
    print("="*70)
    print("\nRecombination frequencies observed:")
    print(f"  Genes A-B: {rf_AB*100:.1f}%")
    print(f"  Genes A-C: {rf_AC*100:.1f}%")
    print(f"  Genes B-C: {rf_BC*100:.1f}%")
    print("\nQuestions:")
    print("1. What is the correct gene order?")
    print("2. What are the map distances between adjacent genes?")
    print("3. Are these genes likely on the same chromosome?")
    print("\n(Run the next cell to see the solution)\n")
    
    return {
        'AB': rf_AB,
        'AC': rf_AC,
        'BC': rf_BC
    }, positions

# Generate practice problems
problem_data, true_positions = create_practice_problem(
    "Fish Genetic Markers",
    "Three microsatellite markers were tested in a breeding population of Labeo rohita."
)

In [ ]:
# Solution to practice problem
print("\n" + "="*70)
print("SOLUTION")
print("="*70)

solution = determine_gene_order_from_rf(problem_data)

print(f"\n1. Gene order: {' → '.join(solution['order'])}")
print(f"\n2. Map distances:")
print(f"   {solution['order'][0]}-{solution['order'][1]}: {solution['distances'][0]:.1f} cM")
print(f"   {solution['order'][1]}-{solution['order'][2]}: {solution['distances'][1]:.1f} cM")
print(f"   Total: {solution['total_distance']:.1f} cM")

print(f"\n3. Linkage analysis:")
print(f"   Observed RF between outer genes: {solution['observed_outer_rf']*100:.1f}%")
print(f"   Expected RF if additive: {solution['expected_outer_rf']*100:.1f}%")

if solution['additivity_check']:
    print("   ✓ CONCLUSION: Genes are LINKED (on same chromosome)")
    print("   ✓ The additivity relationship holds!")
else:
    print("   ✗ CONCLUSION: Genes may be UNLINKED or very far apart")
    print("   ✗ Additivity breaks down due to multiple crossovers")

# Draw the solution map
fig = draw_gene_map(solution['order'], solution['distances'], 
                    'Solution: Gene Order and Map Distances')
plt.savefig('practice_solution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*70)

## Summary: Key Concepts

### 1. Poisson Distribution Foundation
- Crossovers are rare, random, independent events
- The **shape** of the Poisson distribution controls genetic outcomes
- Odd crossovers → recombinant gametes
- Even crossovers → parental gametes

### 2. The 50% Limit
**Two different causes:**
- **Linked but distant**: Multiple crossovers balance odd/even → approaches 50%
- **Unlinked**: Independent assortment → exactly 50%

### 3. Three-Point Crosses
**Distinguish linkage from independence:**
- Linked genes: RF(AC) ≈ RF(AB) + RF(BC)
- Unlinked genes: All RFs ≈ 50%, additivity breaks down

### 4. Gene Ordering Algorithm
1. Find maximum RF → outer genes
2. Remaining gene is in the middle
3. Verify with additivity check
4. Draw linear map with distances

---

**For Pattern Hunters:**
This demonstrates how:
- Mathematical distributions (Poisson) create biological constraints (50% limit)
- The same outcome (50% RF) can arise from different mechanisms
- Strategic experimental design (three-point) resolves ambiguity
- Data patterns reveal hidden structures (gene order)

## Extension Activities

### For Students:
1. Modify the map distances and observe how RF changes
2. Create your own three-point cross problems
3. Explore what happens with 4 or 5 genes

### For Teachers:
1. Use local species examples (Western Odisha fauna)
2. Connect to population genetics applications
3. Link to QTL mapping and breeding programs

### Research Extensions:
1. Compare Haldane vs. Kosambi mapping functions
2. Investigate interference and coefficient of coincidence
3. Apply to earthworm genomics data from mining regions